# Лабораторная 03. Ключевые точки и выравнивание лица

Цель: пройти от пяти размеченных точек к двум обученным CNN и выравниванию лица. Сначала реализуйте NME и геометрию на контролируемом снимке, затем на реальных лицах MTFL — аугментацию, гауссовы тепловые карты и декодер, обучите регрессию и карты, выберите модель по val и оцените на официальном test. Нужен CPU; полная работа ориентировочно 2–3 часа плюс обсуждение результатов.

Маршрут: 1) три TODO по метрике и геометрии; 2) контролируемые val/test на одном снимке; 3) три TODO по отражению, тепловым картам и softargmax; 4) реальный MTFL train/val, пилот и 10 эпох для двух архитектур; 5) выбор по val, одно открытие официального test и разбор позы/очков/скорости; 6) выравнивание реальных кадров. В каждом блоке теория и пример находятся рядом с заданием.

Фотография NASA (общественное достояние) и набор MTFL авторов TCDCN загружаются во временный кэш с проверкой SHA-256. Для опыта с одним снимком при отсутствии сети доступен учебный рисунок; реальная часть требует MTFL ZIP или локальный путь FACE_LAB_MTFL_PATH. Снимки, ZIP, личные фотографии и веса не добавлять в Git.

Протокол: первые 16+16 случаев — повороты ОДНОГО снимка и только проверка чувствительности геометрии. Реальные модели обучаются на 2000 кадрах из официального training.txt, валидируются на других 400 из него, сравниваются с mean-shape; 500 кадров официального testing.txt читаются только после выбора модели. Источники train и test отличаются, личности не гарантированы непересекающимися. Числа этой учебной выборки не сравнивайте с опубликованными benchmark без одинакового протокола.

In [ ]:
import os, time, hashlib, tempfile, urllib.request
from pathlib import Path
import numpy as np, cv2, matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)
print("numpy", np.__version__, "OpenCV", cv2.__version__, "CPU")

In [ ]:
# NASA / Eileen Collins, via scikit-image 0.25.2. Manual points are approximate.
URL = "https://raw.githubusercontent.com/scikit-image/scikit-image/v0.25.2/skimage/data/astronaut.png"
SHA = "88431cd9653ccd539741b555fb0a46b61558b301d4110412b5bc28b5e3ea6cb5"
GT = np.array([[201,101],[248,103],[225,123],[206,144],[247,144]], dtype=float)
TARGET = np.array([[42,49],[86,49],[64,73],[47,97],[81,97]], dtype=float)
NAMES = ["левый глаз на изображении","правый глаз","нос","левый угол рта","правый угол рта"]
def fallback():
    im=np.full((512,512,3),(234,239,245),np.uint8)
    cv2.ellipse(im,(225,125),(78,100),0,0,360,(235,204,181),-1)
    for x,y in GT[:2].astype(int):
        cv2.ellipse(im,(x,y),(12,6),0,0,360,(255,255,255),-1)
        cv2.circle(im,(x,y),4,(42,64,86),-1)
    cv2.circle(im,tuple(GT[2].astype(int)),4,(142,100,90),-1)
    cv2.line(im,tuple(GT[3].astype(int)),tuple(GT[4].astype(int)),(145,70,75),3)
    return im
def load_photo():
    local=os.getenv("FACE_LAB_ASTRONAUT_PATH")
    path=Path(local) if local else Path(tempfile.gettempdir())/"applied_ai_face_lab"/"astronaut.png"
    if not path.exists() and not local:
        try:
            path.parent.mkdir(parents=True,exist_ok=True)
            with urllib.request.urlopen(URL,timeout=20) as r: raw=r.read()
            if hashlib.sha256(raw).hexdigest()!=SHA: raise ValueError("SHA-256 mismatch")
            path.write_bytes(raw)
        except Exception as e:
            print("Снимок недоступен; учебный рисунок:",str(e)[:120])
            return fallback(),False
    raw=path.read_bytes()
    if hashlib.sha256(raw).hexdigest()!=SHA: raise ValueError("SHA-256 mismatch")
    image=cv2.imdecode(np.frombuffer(raw,np.uint8),cv2.IMREAD_COLOR)
    if image is None or image.shape[:2]!=(512,512): raise ValueError("Требуется снимок 512×512")
    return cv2.cvtColor(image,cv2.COLOR_BGR2RGB),True
PHOTO,IS_REAL=load_photo()
print("Источник:", "NASA / Eileen Collins" if IS_REAL else "учебный рисунок")

In [ ]:
def show_points(ax,image,points,title,marker="o",color="orange"):
    ax.imshow(image)
    p=np.asarray(points)
    ax.scatter(p[:,0],p[:,1],s=55,c=color,marker=marker,edgecolors="black")
    for i,(x,y) in enumerate(p): ax.text(x+4,y-3,str(i),color="white",fontsize=9,
        bbox=dict(facecolor="black",alpha=.6,pad=1))
    ax.set_title(title);ax.axis("off")
def eye_transform(src,dst):
    # Similarity determined by two eye centers.
    src=np.asarray(src,float);dst=np.asarray(dst,float)
    u=src[1]-src[0];v=dst[1]-dst[0]
    if np.linalg.norm(u)<1e-8: raise ValueError("Глаза совпали")
    s=np.linalg.norm(v)/np.linalg.norm(u)
    a=np.arctan2(v[1],v[0])-np.arctan2(u[1],u[0])
    R=np.array([[np.cos(a),-np.sin(a)],[np.sin(a),np.cos(a)]])
    A=s*R;t=dst[0]-A@src[0]
    return np.column_stack([A,t])
def move(points,M):
    p=np.asarray(points,float)
    return p@M[:,:2].T+M[:,2]
def crop(image,M):
    return cv2.warpAffine(image,M,(128,128),borderValue=(240,240,240))

## Блок 1. NME и маска точек

NME = сумма евклидовых ошибок по включённым точкам / (число включённых точек × межглазное расстояние GT). Маска оценки меняет состав среднего, но не нормирующее расстояние между точками 0 и 1. Если одна из пяти точек сдвинута ровно на межглазное расстояние, NME=1/5=0.2; при маске только этой точки NME=1. Пустая маска и совпавшие глаза — ошибка входных данных. Сначала запишите оба ожидаемых числа, затем реализуйте TODO и выполните тесты.

In [ ]:
def normalized_mean_error(pred,gt,visible=None):
    # TODO 1: массивы (N,2), конечность, d_eye между GT 0 и 1,
    # непустая булева маска и средняя ошибка выбранных точек.
    raise NotImplementedError("TODO 1: normalized_mean_error")

In [ ]:
toy=np.array([[0.,0.],[10.,0.],[5.,4.],[2.,7.],[8.,7.]])
pred=toy.copy();pred[2,0]+=10
assert np.isclose(normalized_mean_error(pred,toy),.2)
assert np.isclose(normalized_mean_error(pred,toy,[False,False,True,False,False]),1.)
assert np.isclose(normalized_mean_error(toy,toy),0.)
for args in ((toy[:1],toy[:1]),(toy,toy,np.zeros(5,dtype=bool))):
    try: normalized_mean_error(*args)
    except ValueError: pass
    else: raise AssertionError("Ожидается ValueError")
print("NME: открытые проверки пройдены")

Собственный контроль 1: придумайте две ошибочные точки и маску, запишите ожидаемое число до выполнения, затем сравните с функцией и объясните результат.

## Блок 2. Взвешенный similarity transform

Минимизируйте сумму w_i ||sR src_i + t − dst_i||². Веса неотрицательны, нулевой вес исключает точку; R — поворот без отражения. Мини-пример: исходные (0,0),(10,0) переходят в (2,3),(22,3), значит масштаб 2, поворот 0°, перенос (2,3). Подсказка для общего случая: взвешенные центры, ковариация, SVD, коррекция отражения и масштаб. Верните 2×3 матрицу для OpenCV; проверьте вырожденные входы. Сначала выпишите ожидаемую матрицу мини-примера.

In [ ]:
def weighted_similarity(src,dst,weights=None):
    # TODO 2: минимум суммы w_i ||sR src_i+t-dst_i||^2.
    # Взвешенные центры, SVD, коррекция отражения, масштаб, перенос.
    # Вернуть матрицу 2x3. Нулевой вес исключает точку.
    raise NotImplementedError("TODO 2: weighted_similarity")

In [ ]:
src=np.array([[0.,0.],[2.,0.],[0.,2.],[2.,2.]])
A=np.array([[0.,-1.],[1.,0.]])*1.5
t=np.array([4.,-2.]);dst=src@A.T+t
M=weighted_similarity(src,dst)
assert M.shape==(2,3) and np.allclose(M[:,:2],A,atol=1e-8)
assert np.allclose(M[:,2],t,atol=1e-8)
out=dst.copy();out[2]=[100.,100.]
assert np.allclose(weighted_similarity(src,out,[1,1,0,1]),M,atol=1e-8)
for w in ([0,0,0,0],[1,-1,1,1]):
    try: weighted_similarity(src,dst,w)
    except ValueError: pass
    else: raise AssertionError("Ожидается ValueError")
print("Similarity: открытые проверки пройдены")

Собственный контроль 2: задайте известный масштаб и перенос для 3–5 точек, выведите матрицу вручную и сравните. Затем добавьте выброс с нулевым весом.

## Блок 3. Перенос точек и кроп

Одна матрица должна переносить и пиксели изображения, и точки: [x′,y′] = M[x,y,1]. Размеры входов N×2 и 2×3. Например, M=[[2,0,3],[0,2,−1]] переводит (1,4) в (5,7). Сначала проверьте этот расчёт вручную. Ошибка перепутанной системы координат испортит результат, даже если кроп выглядит правдоподобно.

In [ ]:
def transform_points(points,M):
    # TODO 3: перенести (N,2) матрицей 2x3, сохранив порядок.
    # Проверить формы и конечность входа.
    raise NotImplementedError("TODO 3: transform_points")

In [ ]:
assert np.allclose(transform_points(src,M),dst,atol=1e-8)
assert transform_points(np.empty((0,2)),M).shape==(0,2)
try: transform_points(np.zeros((3,3)),M)
except ValueError: pass
else: raise AssertionError("Ожидается ValueError")
print("Точки: открытые проверки пройдены")

Собственный контроль 3: преобразуйте угол единичного квадрата известной матрицей и сравните с ручным расчётом.

## Опыт: закрытый нос

В половине 16 сценариев нос смещён на (+18,−14) пикселей. Остальные точки имеют небольшой гауссов шум. Val и test созданы разными seed. До расчёта запишите гипотезу: как понижение веса носа меняет ошибку выравнивания и почему может повредить при корректном носе. Метрика — средняя ошибка ИСТИННЫХ точек в целевом кропе 128×128, px.

In [ ]:
def make_cases(count,seed):
    # Один снимок, контролируемая геометрия; это не benchmark детектора.
    rng=np.random.default_rng(seed);cases=[]
    for i in range(count):
        angle=float(rng.uniform(-22,22));scale=float(rng.uniform(.88,1.12))
        M=cv2.getRotationMatrix2D((256,256),angle,scale)
        M[:,2]+=rng.uniform(-18,18,size=2)
        image=cv2.warpAffine(PHOTO,M,(512,512),borderValue=(240,240,240))
        true=transform_points(GT,M)
        estimated=true+rng.normal(0,1.8,size=true.shape)
        occluded=i<count//2
        if occluded: estimated[2]+=[18,-14]
        cases.append((image,true,estimated,occluded))
    return cases
def one_case(case,nose_weight):
    image,true,estimated,occluded=case
    weights=[1,1,nose_weight if occluded else 1,.5,.5]
    M=weighted_similarity(estimated,TARGET,weights)
    aligned_true=transform_points(true,M)
    error=float(np.linalg.norm(aligned_true-TARGET,axis=1).mean())
    return error,M
def summary(cases,weight):
    errors=[];times=[]
    for case in cases:
        start=time.perf_counter();err,_=one_case(case,weight)
        times.append((time.perf_counter()-start)*1000);errors.append(err)
    return np.mean(errors),np.median(errors),np.mean(times)
VAL=make_cases(16,4104);TEST=make_cases(16,9104)

In [ ]:
CANDIDATES=[0.,.1,.25,.5,1.]
rows=[]
for weight in CANDIDATES:
    row=(weight,*summary(VAL,weight));rows.append(row)
print("Val: вес носа | mean error px | median error px | CPU ms/case")
for row in rows: print("%8.2f | %13.3f | %15.3f | %11.3f"%row)
print("Минимум mean на val:",min(rows,key=lambda r:r[1])[0])

Запишите выбор веса и основание по таблице val до открытия test. Можно выбрать минимум или более простое правило, если различия малы. Не настраивайте вес после чтения test. Укажите цену по ошибке и затратам CPU.

In [ ]:
SELECTED_NOSE_WEIGHT=None  # TODO: зафиксировать после анализа val
RUN_TEST=False  # True только после записи выбора в отчёте
if RUN_TEST:
    if SELECTED_NOSE_WEIGHT not in CANDIDATES: raise ValueError("Зафиксируйте вес")
    print("TEST baseline w=1",summary(TEST,1.))
    print("TEST selected",summary(TEST,SELECTED_NOSE_WEIGHT))
else:
    print("Test закрыт: сначала решение по val")

## Визуальный контроль одинаковых случаев

Ниже два фиксированных случая: с закрытым носом и без. Голубые кружки показывают целевые места, оранжевые — истинные точки после кропа. Назовите, какой участок сдвинулся сильнее и почему.

In [ ]:
for idx in (0,12):
    image,true,estimated,occ=VAL[idx]
    print("Сценарий",idx,"закрытый нос:",occ,
          "NME по всем точкам:",round(normalized_mean_error(estimated,true),3))
    fig,ax=plt.subplots(1,3,figsize=(11,4))
    show_points(ax[0],image,estimated,"Оценённые точки")
    for j,w in enumerate((1.,0.),start=1):
        err,M=one_case(VAL[idx],w)
        out=cv2.warpAffine(image,M,(128,128),borderValue=(240,240,240))
        after=transform_points(true,M)
        show_points(ax[j],out,after,f"Вес носа {w:g}; ошибка {err:.2f}px")
        ax[j].scatter(TARGET[:,0],TARGET[:,1],s=85,facecolors="none",edgecolors="cyan")
    plt.tight_layout();plt.show()

## От учебной геометрии к реальным лицам: MTFL

Официальный Multi-Task Facial Landmark (MTFL) содержит 10 000 train и 2995 test изображений с пятью точками: центры глаз, нос, углы рта. В файле разметки координаты даны блоками x1…x5, затем y1…y5; дальше пол, улыбка, очки и поза (1/5 — профиль, 3 — фронт). Источник и цитирование: [страница авторов](https://mmlab.ie.cuhk.edu.hk/projects/TCDCN.html), [статья ECCV 2014](https://home.ie.cuhk.edu.hk/~ccloy/files/eccv_2014_deepfacealign.pdf). Архив ~150 MB хранится только во временном каталоге, сверяется с указанным SHA-256 и не попадает в репозиторий.

Мы делаем воспроизводимый небольшой опыт: seed=4104, 2000 train + 400 val из training.txt, затем 500 из официального testing.txt. Предобработка — grayscale и прямое растяжение каждого кадра до 64×64. Координаты переводятся в [0,1] по исходным размерам; NME и PCK считаются **в координатах растянутого кадра** и нормируются межглазным расстоянием GT в этом же пространстве. Это намеренно простой baseline с собственной геометрией и подвыборкой, не число стандартного MTFL benchmark. Запишите ожидаемое направление разницы между val и test: источники снимков отличаются, а разбивка не удостоверяет раздельные личности.

In [ ]:
import copy, zipfile, urllib.request
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
torch.set_num_threads(min(4,os.cpu_count() or 1))
print("PyTorch",torch.__version__,"CPU threads",torch.get_num_threads())

## Блок 4. Парная аугментация

Горизонтальное отражение меняет x на 1−x и переставляет семантические пары: левый/правый глаз и угол рта. Если отразить только изображение, модель получит неверную метку. Для точки (0.10,0.20) левого глаза после отражения положение (0.90,0.20), а её имя становится «правый глаз». Два отражения должны вернуть исходные изображение и порядок точек. Реализуйте TODO 4, не изменяя аргументы; он будет использован в обеих моделях.

In [ ]:
def flip_sample(image, points):
    # TODO 4: отразить H×W изображение и пять нормированных точек.
    # x' = 1 - x; переставить глаза и углы рта [1,0,2,4,3].
    # Не менять исходные массивы.
    raise NotImplementedError("TODO 4: flip_sample")

In [ ]:
toy_im=np.arange(48,dtype=np.uint8).reshape(6,8)
toy_pts=np.array([[.1,.2],[.8,.3],[.4,.6],[.2,.8],[.9,.7]])
im_copy=toy_im.copy();pts_copy=toy_pts.copy()
flipped,fp=flip_sample(toy_im,toy_pts)
assert flipped.shape==toy_im.shape and np.array_equal(flipped[:,0],toy_im[:,-1])
assert np.allclose(fp[0],[.2,.3]) and np.allclose(fp[3],[.1,.7])
twice,back=flip_sample(flipped,fp)
assert np.array_equal(twice,toy_im) and np.allclose(back,toy_pts)
assert np.array_equal(toy_im,im_copy) and np.array_equal(toy_pts,pts_copy)
print("Отражение: открытые проверки пройдены")

## Блок 5. Целевые карты и дифференцируемый декодер

Сеть карт выдаёт по сетке 16×16 пять полей логитов. Для точки (0.5,0.5) центр гауссианы при размере 9 лежит в ячейке (4,4); сумма вероятностей каждой карты равна 1. При обучении минимизируем кросс-энтропию к нормированной гауссиане, так что модель сохраняет пространственную структуру. Реализуйте TODO 5. Крайние точки тоже должны давать корректную нормированную карту.

In [ ]:
def gaussian_heatmaps(points, size=16, sigma=1.2):
    # TODO 5: точки B×5×2 в [0,1] → B×5×size×size.
    # Центры сетки 0...size-1; гауссианы нормировать до суммы 1 по каждой карте.
    # Сохранять device и dtype PyTorch.
    raise NotImplementedError("TODO 5: gaussian_heatmaps")

In [ ]:
q=torch.tensor([[[0.,0.],[1.,1.],[.5,.5],[.25,.75],[.75,.25]]],dtype=torch.float32)
maps=gaussian_heatmaps(q,size=9,sigma=.8)
assert maps.shape==(1,5,9,9) and torch.allclose(maps.sum((-2,-1)),torch.ones(1,5))
assert torch.argmax(maps[0,0]).item()==0
assert torch.argmax(maps[0,1]).item()==80
assert torch.argmax(maps[0,2]).item()==40
assert torch.isfinite(maps).all()
try: gaussian_heatmaps(q,size=9,sigma=0)
except ValueError: pass
else: raise AssertionError("sigma=0 должен вызывать ValueError")
try: gaussian_heatmaps(q,size=9,sigma=float("nan"))
except ValueError: pass
else: raise AssertionError("sigma=nan должен вызывать ValueError")
print("Тепловые карты: открытые проверки пройдены")

Softargmax сначала применяет пространственный softmax, затем считает математическое ожидание координаты по сетке. Пик в (x=4,y=1) на сетке 6×4 даёт (4/5,1/3). Два равных пика усредняются; argmax потерял бы эту информацию. Реализуйте TODO 6; температура управляет остротой, но в сравнении моделей используйте фиксированную температуру 1.

In [ ]:
def decode_heatmaps(logits, temperature=1.):
    # TODO 6: пространственный softmax по каждому каналу и ожидаемая координата.
    # Вход B×5×H×W, выход B×5×2 в [0,1]; порядок x,y.
    # temperature>0, H/W>=2; не использовать argmax в учебном декодере.
    raise NotImplementedError("TODO 6: decode_heatmaps")

In [ ]:
peak=torch.full((1,5,4,6),-100.,dtype=torch.float32)
peak[0,:,1,4]=100.
xy=decode_heatmaps(peak)
assert xy.shape==(1,5,2) and torch.allclose(xy[0,0],torch.tensor([4/5,1/3]),atol=1e-4)
uniform=decode_heatmaps(torch.zeros(2,5,4,6))
assert torch.allclose(uniform,torch.full((2,5,2),.5),atol=1e-6)
two=torch.full((1,5,4,6),-100.)
two[0,:,1,1]=100.;two[0,:,1,5]=100.
assert torch.allclose(decode_heatmaps(two)[0,0],torch.tensor([.6,1/3]),atol=1e-4)
try: decode_heatmaps(peak,temperature=0)
except ValueError: pass
else: raise AssertionError("temperature=0 должен вызывать ValueError")
try: decode_heatmaps(peak,temperature=float("nan"))
except ValueError: pass
else: raise AssertionError("temperature=nan должен вызывать ValueError")
print("Декодер: открытые проверки пройдены")

## Данные и визуальный контроль разметки

Загрузка читает только выбранные JPEG из ZIP, без распаковки в Git. Перед обучением проверьте порядок пяти точек на трёх реальных лицах. Обратите внимание: видимая/закрытая точка в MTFL не сопровождается отдельной маской видимости; модель учит пять координат для каждого кадра. В реальных системах протокол закрытых точек надо уточнять отдельно.

In [ ]:
MTFL_URL="https://mmlab.ie.cuhk.edu.hk/projects/TCDCN/data/MTFL.zip"
MTFL_SHA="d15bd336ede1349d342f9dc27377cdb7954bc7d9ec5f3565c64d2b7575445a51"
MTFL_BYTES=150152960
MTFL_PATH=Path(os.getenv("FACE_LAB_MTFL_PATH") or
               Path(tempfile.gettempdir())/"applied_ai_face_lab"/"MTFL.zip")
IMAGE_SIZE=64
SEED=4104
N_TRAIN,N_VAL,N_TEST=2000,400,500

def sha256_file(path):
    digest=hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda:f.read(1<<20),b""):
            digest.update(chunk)
    return digest.hexdigest()

def ensure_mtfl():
    if MTFL_PATH.exists():
        if MTFL_PATH.stat().st_size!=MTFL_BYTES or sha256_file(MTFL_PATH)!=MTFL_SHA:
            raise ValueError("MTFL ZIP: размер или SHA-256 не совпадает; не используйте этот файл")
        return MTFL_PATH
    if os.getenv("FACE_LAB_MTFL_PATH"):
        raise FileNotFoundError("Путь FACE_LAB_MTFL_PATH не найден")
    MTFL_PATH.parent.mkdir(parents=True,exist_ok=True)
    part=MTFL_PATH.with_suffix(".part")
    written=0
    try:
        print("Загружается официальный MTFL ZIP (~150 MB) во временный кэш...")
        with urllib.request.urlopen(MTFL_URL,timeout=60) as source, part.open("wb") as target:
            while True:
                chunk=source.read(1<<20)
                if not chunk:break
                written+=len(chunk)
                if written>MTFL_BYTES:raise ValueError("MTFL больше ожидаемого размера")
                target.write(chunk)
                if written//(25<<20)!=(written-len(chunk))//(25<<20):
                    print("Загружено",round(written/1e6),"MB")
        if written!=MTFL_BYTES or sha256_file(part)!=MTFL_SHA:
            raise ValueError("MTFL ZIP: размер или SHA-256 не совпадает")
        os.replace(part,MTFL_PATH)
    finally:
        if part.exists():part.unlink()
    return MTFL_PATH

def mtfl_rows(archive,name):
    rows=[]
    for line in archive.read(name).decode("utf-8").splitlines():
        if not line.strip():continue
        fields=line.split()
        if len(fields)!=15:raise ValueError("Нужны path, 10 чисел и 4 атрибута")
        member=fields[0].replace("\\","/")
        if member.startswith("/") or ".." in Path(member).parts:
            raise ValueError("Недопустимый путь внутри архива")
        # Формат MTFL: сначала x1...x5, затем y1...y5.
        points=np.array([float(v) for v in fields[1:11]],np.float32).reshape(2,5).T
        attrs=tuple(int(v) for v in fields[11:])
        if not np.isfinite(points).all() or attrs[-1] not in (1,2,3,4,5):
            raise ValueError("Некорректная разметка MTFL")
        rows.append((member,points,attrs))
    return rows

def load_mtfl(archive,rows,indices):
    images=[];labels=[];poses=[];glasses=[]
    for index in indices:
        member,points,attrs=rows[int(index)]
        raw=np.frombuffer(archive.read(member),np.uint8)
        image=cv2.imdecode(raw,cv2.IMREAD_GRAYSCALE)
        if image is None:raise ValueError("JPEG не прочитан: "+member)
        h,w=image.shape
        norm=points/np.array([w-1,h-1],dtype=np.float32)
        if np.any(norm<0) or np.any(norm>1):
            raise ValueError("Точка вне изображения: "+member)
        images.append(cv2.resize(image,(IMAGE_SIZE,IMAGE_SIZE),
                                 interpolation=cv2.INTER_AREA))
        labels.append(norm)
        glasses.append(attrs[2]);poses.append(attrs[3])
    return (np.stack(images).astype(np.uint8),
            np.stack(labels).astype(np.float32),
            np.asarray(poses,np.int64),np.asarray(glasses,np.int64))

mtfl_file=ensure_mtfl()
with zipfile.ZipFile(mtfl_file) as z:
    train_rows=mtfl_rows(z,"training.txt")
    assert len(train_rows)==10000
    permutation=np.random.default_rng(SEED).permutation(len(train_rows))
    train_ids=permutation[:N_TRAIN];val_ids=permutation[N_TRAIN:N_TRAIN+N_VAL]
    X_TRAIN,Y_TRAIN,POSE_TRAIN,GLASSES_TRAIN=load_mtfl(z,train_rows,train_ids)
    X_VAL,Y_VAL,POSE_VAL,GLASSES_VAL=load_mtfl(z,train_rows,val_ids)
print("MTFL: train",len(X_TRAIN),"val",len(X_VAL),
      "из официального training.txt; test пока не читали")
print("Позы val:",{p:int(np.sum(POSE_VAL==p)) for p in range(1,6)})
fig,ax=plt.subplots(1,3,figsize=(9,3))
for k,a in enumerate(ax):
    a.imshow(X_TRAIN[k],cmap="gray",vmin=0,vmax=255)
    a.scatter(Y_TRAIN[k,:,0]*(IMAGE_SIZE-1),Y_TRAIN[k,:,1]*(IMAGE_SIZE-1),
              c="orange",s=25,edgecolors="black")
    a.set(title=f"MTFL train; поза {POSE_TRAIN[k]}",xlim=(0,IMAGE_SIZE),
          ylim=(IMAGE_SIZE,0))
    a.axis("off")
plt.tight_layout();plt.show()

In [ ]:
example=torch.from_numpy(Y_TRAIN[:1].copy()).float()
target_map=gaussian_heatmaps(example,size=16,sigma=1.2)[0]
fig,ax=plt.subplots(1,2,figsize=(6,3))
ax[0].imshow(X_TRAIN[0],cmap="gray",vmin=0,vmax=255)
ax[0].scatter(Y_TRAIN[0,:,0]*63,Y_TRAIN[0,:,1]*63,c="orange",s=18)
ax[0].set_title("Лицо и 5 ориентиров")
ax[1].imshow(target_map[2].numpy(),cmap="magma")
ax[1].set_title("Целевая карта носа 16×16")
for a in ax:a.axis("off")
plt.tight_layout();plt.show()

## Две архитектуры на одной выборке

| Ветка | Представление | Выход | Потеря | Декодирование |
|---|---|---|---|---|
| Координатная | 4 свёрточных блока с уменьшением 64→32→16→8→4, общие признаки | 10 чисел в [0,1] | Smooth L1 | reshape в 5×2 |
| Тепловая | 3 уменьшения до 8, затем признаки и upsample до 16 | 5 карт 16×16 | Кросс-энтропия к гауссианам | softargmax |

Число параметров, секунды обучения и CPU миллисекунды на один кадр выводятся при запуске. Валидация сравнивает модели **одной** метрикой, хотя функции потерь различаются. Mean-shape baseline всегда предсказывает среднее расположение точек из train. Режим train использует одинаковое отражение/яркость для обеих моделей; val и test без аугментации. Пройдите 2-эпоховый пилот, проверьте формы выходов и снижение потерь, затем включите полный опыт. Пилот не включать в итоговую таблицу.

In [ ]:
class FaceDataset(Dataset):
    def __init__(self, images, points, augment=False):
        self.images=images;self.points=points;self.augment=augment
    def __len__(self):return len(self.images)
    def __getitem__(self,i):
        image=self.images[i].copy();points=self.points[i].copy()
        if self.augment and np.random.random()<.5:
            image,points=flip_sample(image,points)
        if self.augment:
            image=np.clip(image.astype(np.float32)*np.random.uniform(.9,1.1),0,255).astype(np.uint8)
        x=torch.from_numpy(np.ascontiguousarray(image)).float()[None]/255.
        y=torch.from_numpy(np.ascontiguousarray(points)).float()
        return x,y

class CoordinateNet(nn.Module):
    def __init__(self):
        super().__init__()
        channels=[1,16,32,64,96];layers=[]
        for a,b in zip(channels[:-1],channels[1:]):
            layers += [nn.Conv2d(a,b,3,stride=2,padding=1),
                       nn.BatchNorm2d(b),nn.ReLU(inplace=True)]
        self.features=nn.Sequential(*layers,nn.AdaptiveAvgPool2d((2,2)))
        self.head=nn.Sequential(nn.Flatten(),nn.Linear(96*4,64),nn.ReLU(),
                                nn.Linear(64,10),nn.Sigmoid())
    def forward(self,x):return self.head(self.features(x)).reshape(-1,5,2)

class HeatmapNet(nn.Module):
    def __init__(self):
        super().__init__()
        channels=[1,16,32,64,96];layers=[]
        for a,b in zip(channels[:-1],channels[1:]):
            layers += [nn.Conv2d(a,b,3,stride=2 if a in (1,16,32) else 1,padding=1),
                       nn.BatchNorm2d(b),nn.ReLU(inplace=True)]
        self.features=nn.Sequential(*layers)
        self.head=nn.Sequential(nn.Upsample(size=(16,16),mode="bilinear",align_corners=False),
                                nn.Conv2d(96,64,3,padding=1),nn.ReLU(),
                                nn.Conv2d(64,5,1))
    def forward(self,x):return self.head(self.features(x))

def per_face_nme(pred,gt):
    pred=np.asarray(pred);gt=np.asarray(gt)
    if pred.shape!=gt.shape or pred.ndim!=3 or pred.shape[1:]!=(5,2):
        raise ValueError("Нужны одинаковые N×5×2")
    eye=np.linalg.norm(gt[:,0]-gt[:,1],axis=1)
    if np.any(eye<1e-8):raise ValueError("Нулевое межглазное расстояние")
    point_error=np.linalg.norm(pred-gt,axis=2)/eye[:,None]
    return point_error.mean(axis=1),point_error

def model_metrics(pred,gt):
    errors,per_point=per_face_nme(pred,gt)
    return {"nme":float(np.mean(errors)),"p90":float(np.percentile(errors,90)),
            "pck10":float(np.mean(per_point<=.10)),
            "pck20":float(np.mean(per_point<=.20))}

@torch.inference_mode()
def predict_points(model,kind,images):
    model.eval();pieces=[]
    for i in range(0,len(images),64):
        x=torch.from_numpy(images[i:i+64].copy()).float()[:,None]/255.
        output=model(x)
        if kind=="heatmap":output=decode_heatmaps(output)
        pieces.append(output.cpu().numpy())
    return np.concatenate(pieces).astype(np.float32)

def train_model(kind, train_images,train_points,val_images,val_points,
                epochs=10,augment=True,seed=SEED):
    if kind not in ("coordinate","heatmap"):raise ValueError("Неизвестная модель")
    torch.manual_seed(seed);np.random.seed(seed)
    model=CoordinateNet() if kind=="coordinate" else HeatmapNet()
    optim=torch.optim.Adam(model.parameters(),lr=.002)
    loader=DataLoader(FaceDataset(train_images,train_points,augment=augment),
                      batch_size=64,shuffle=True,num_workers=0,
                      generator=torch.Generator().manual_seed(seed))
    history=[];best=float("inf");best_state=None
    started=time.perf_counter()
    for epoch in range(1,epochs+1):
        model.train();total=0.
        for x,y in loader:
            optim.zero_grad(set_to_none=True)
            output=model(x)
            if kind=="coordinate":
                loss=F.smooth_l1_loss(output,y,beta=.02)
            else:
                target=gaussian_heatmaps(y,size=16,sigma=1.2)
                logprob=F.log_softmax(output.flatten(2),dim=-1)
                loss=-(target.flatten(2)*logprob).sum(dim=-1).mean()
            loss.backward();optim.step();total+=float(loss.detach())*len(x)
        val_pred=predict_points(model,kind,val_images)
        score=model_metrics(val_pred,val_points)["nme"]
        history.append((epoch,total/len(train_images),score))
        if score<best:
            best=score;best_state=copy.deepcopy(model.state_dict())
        print(f"{kind:10s} эпоха {epoch:2d}: loss={history[-1][1]:.4f}, val NME={score:.4f}")
    model.load_state_dict(best_state)
    elapsed=time.perf_counter()-started
    return model,np.asarray(history,float),elapsed

def cpu_forward_ms(model,kind,repeats=30):
    model.eval();x=torch.from_numpy(X_VAL[:1].copy()).float()[:,None]/255.
    with torch.inference_mode():
        for _ in range(10):
            out=model(x)
            if kind=="heatmap":decode_heatmaps(out)
        samples=[]
        for _ in range(repeats):
            t=time.perf_counter();out=model(x)
            if kind=="heatmap":decode_heatmaps(out)
            samples.append((time.perf_counter()-t)*1000)
    return float(np.median(samples))

def metric_row(name,pred,gt,ms=None):
    row=model_metrics(pred,gt)
    row["model"]=name;row["ms"]=ms
    return row

def print_metric_table(rows):
    print("модель         NME     p90   PCK@.10 PCK@.20 CPU ms/лицо")
    for r in rows:
        speed="—" if r["ms"] is None else f'{r["ms"]:.2f}'
        print(f'{r["model"]:12s} {r["nme"]:.3f}   {r["p90"]:.3f}   '
              f'{r["pck10"]:.3f}    {r["pck20"]:.3f}      {speed}')

In [ ]:
RUN_PILOT=True  # Короткая проверка, что потери уменьшаются; test не используется
if RUN_PILOT:
    print("Пилот: 256 train, 80 val, 2 эпохи; значения НЕ для итоговой таблицы")
    pilot_models={}
    for kind in ("coordinate","heatmap"):
        pilot_models[kind]=train_model(kind,X_TRAIN[:256],Y_TRAIN[:256],
                                      X_VAL[:80],Y_VAL[:80],epochs=2,
                                      augment=True,seed=SEED)[0]

## Полный опыт: val без подглядывания в test

Поставьте RUN_FULL=True и запустите 10 эпох для каждой архитектуры. Из истории выберите лучший checkpoint каждой модели по среднему val NME. Таблица выводит NME, p90 ошибок по лицам, PCK@0.10 и PCK@0.20 по точкам, а также медиану CPU latency batch=1. Порог PCK доля **межглазного расстояния** GT, так что 0.10 означает не 10 пикселей. Время измеряет только forward и декодер: это не скорость полной системы с детектором и I/O.

До запуска запишите гипотезу: какая архитектура будет точнее и быстрее? После выполнения запишите таблицу и объясните, согласуются ли ранжирования NME, p90 и PCK. Сравните с mean-shape. Отдельно отметьте эпоху лучшего val checkpoint, даже если последняя эпоха хуже.

In [ ]:
RUN_FULL=False  # Для сдачи установите True; полный опыт ~1–3 мин CPU после загрузки данных
MODEL_FITS={};MODEL_VAL={};MODEL_ROWS=[]
mean_shape=Y_TRAIN.mean(axis=0)
baseline_val=np.repeat(mean_shape[None],len(Y_VAL),axis=0)
MODEL_ROWS.append(metric_row("mean-shape",baseline_val,Y_VAL))
if RUN_FULL:
    for kind in ("coordinate","heatmap"):
        model,history,seconds=train_model(kind,X_TRAIN,Y_TRAIN,X_VAL,Y_VAL,
                                         epochs=10,augment=True,seed=SEED)
        prediction=predict_points(model,kind,X_VAL)
        speed=cpu_forward_ms(model,kind)
        MODEL_FITS[kind]={"model":model,"history":history,"train_seconds":seconds}
        MODEL_VAL[kind]=model_metrics(prediction,Y_VAL)
        MODEL_ROWS.append(metric_row(kind,prediction,Y_VAL,speed))
        print(f"{kind}: {sum(p.numel() for p in model.parameters()):,} параметров; "
              f"обучение {seconds:.1f} с")
    print("ВАЛИДАЦИЯ: общие 400 кадров, PCK нормирован межглазным расстоянием")
    print_metric_table(MODEL_ROWS)
    fig,ax=plt.subplots(1,3,figsize=(13,3.5))
    for j,(kind,color) in enumerate((("coordinate","royalblue"),("heatmap","darkorange"))):
        h=MODEL_FITS[kind]["history"]
        ax[j].plot(h[:,0],h[:,1],"o-",color=color)
        ax[j].set(xlabel="Эпоха",ylabel="Train loss",title=kind)
        ax[2].plot(h[:,0],h[:,2],"o-",label=kind,color=color)
    ax[2].set(xlabel="Эпоха",ylabel="Val NME",title="Общая метрика")
    ax[2].legend();plt.tight_layout();plt.show()
else:
    print("Пока только mean-shape baseline. Для сдачи: RUN_FULL=True и повторный запуск.")
    print_metric_table(MODEL_ROWS)

## Контрольная абляция аугментации

Чтобы отделить эффект представления от эффекта процедуры обучения, повторите только координатную сеть с теми же train/val, seed и 10 эпохами, но без отражения и изменения яркости: RUN_ABLATION=True. Сравните её с исходной координатной сетью по NME/PCK на val и по позам 2, 3, 4. Профильные группы 1 и 5 слишком малы в этой val-подвыборке для уверенного вывода. До запуска запишите направление ожидаемой разницы. После запуска объясните, как интерпретировать ухудшение либо улучшение: аугментация не обязана помогать при каждой выборке и настройке. Test этой абляцией не используется; основной выбор модели остаётся среди двух предзаданных архитектур.

In [ ]:
RUN_ABLATION=False  # Для сдачи True: контрольный опыт без отражения и изменения яркости
ABLATION_VAL={}
if RUN_ABLATION:
    if not RUN_FULL:raise ValueError("Сначала выполните RUN_FULL")
    plain,h_plain,plain_seconds=train_model("coordinate",X_TRAIN,Y_TRAIN,X_VAL,Y_VAL,
                                             epochs=10,augment=False,seed=SEED)
    plain_pred=predict_points(plain,"coordinate",X_VAL)
    regular_pred=predict_points(MODEL_FITS["coordinate"]["model"],"coordinate",X_VAL)
    ABLATION_VAL={"no_augmentation":model_metrics(plain_pred,Y_VAL),
                  "with_augmentation":MODEL_VAL["coordinate"]}
    print("Контрольная абляция координатной модели; общие train/val/эпохи/seed")
    print_metric_table([metric_row("без аугм.",plain_pred,Y_VAL),
                        metric_row("с аугм.",regular_pred,Y_VAL)])
    print(f"Дополнительное обучение: {plain_seconds:.1f} с")
    for pose in (2,3,4):
        mask=POSE_VAL==pose
        a=per_face_nme(plain_pred[mask],Y_VAL[mask])[0].mean()
        b=per_face_nme(regular_pred[mask],Y_VAL[mask])[0].mean()
        print(f"поза {pose}: n={mask.sum()}, без аугм. {a:.3f}, с аугм. {b:.3f}")
else:
    print("Абляция закрыта. Для сдачи RUN_ABLATION=True и отдельная строка вывода.")

## Зафиксируйте решение и откройте официальный test

Выберите `SELECTED_MODEL` по **val NME**, до просмотра test запишите прогноз: NME на test относительно val, поведение профиля и очков, ожидаемое изменение качества crop. Затем установите RUN_MODEL_TEST=True и запустите ячейку один раз. Выводит обе обученные модели и baseline для сравнения, но выбранная модель остаётся зафиксированной. У test другой источник изображений; разницу нельзя объяснить только архитектурой. Малые подгруппы профиля дают шумную оценку; обязательно укажите их размер.

In [ ]:
SELECTED_MODEL=None  # Запишите здесь "coordinate" или "heatmap" после таблицы val
RUN_MODEL_TEST=False  # True один раз после выбора; официальный testing.txt не был в обучении
TEST_RESULTS={}
if RUN_MODEL_TEST:
    if not RUN_FULL or set(MODEL_FITS)!={"coordinate","heatmap"}:
        raise ValueError("Сначала выполните полный опыт на train/val")
    if SELECTED_MODEL not in MODEL_FITS:
        raise ValueError("Выберите модель по val и зафиксируйте выбор")
    print("Зафиксированная модель:",SELECTED_MODEL)
    rng_test=np.random.default_rng(SEED+1)
    with zipfile.ZipFile(mtfl_file) as z:
        test_rows=mtfl_rows(z,"testing.txt")
        assert len(test_rows)==2995
        test_ids=rng_test.permutation(len(test_rows))[:N_TEST]
        X_TEST,Y_TEST,POSE_TEST,GLASSES_TEST=load_mtfl(z,test_rows,test_ids)
    results=[]
    for name in ("mean-shape","coordinate","heatmap"):
        pred=(np.repeat(mean_shape[None],len(X_TEST),axis=0) if name=="mean-shape"
              else predict_points(MODEL_FITS[name]["model"],name,X_TEST))
        TEST_RESULTS[name]={"pred":pred,"metrics":model_metrics(pred,Y_TEST)}
        speed=None if name=="mean-shape" else next(r["ms"] for r in MODEL_ROWS if r["model"]==name)
        results.append(metric_row(name,pred,Y_TEST,speed))
    print("ОФИЦИАЛЬНЫЙ MTFL TEST: фиксированные 500 кадров; это отдельный источник снимков")
    print_metric_table(results)
    selected=TEST_RESULTS[SELECTED_MODEL]["pred"]
    errors,_=per_face_nme(selected,Y_TEST)
    print("NME выбранной модели по позе (1/5 профиль, 3 фронт):")
    for pose in range(1,6):
        mask=POSE_TEST==pose
        if mask.any():print(f"поза {pose}: n={mask.sum():3d}, NME={errors[mask].mean():.3f}")
    print("NME по очкам (1 — есть, 2 — нет):")
    for glasses in (1,2):
        mask=GLASSES_TEST==glasses
        if mask.any():print(f"очки {glasses}: n={mask.sum():3d}, NME={errors[mask].mean():.3f}")
else:
    print("Официальный test пока закрыт: сначала RUN_FULL и выбор по val.")

## Сквозной визуальный результат

На трёх фиксированных test-примерах слева показаны входное лицо, GT и прогноз выбранной сети; справа — crop от mean-shape, координатной и тепловой моделей. Голубые окружности — целевой шаблон, зелёные — GT после переноса той же матрицей. Сравните baseline, координатную и тепловую модели на одних и тех же кадрах при фронтальной позе, профиле и очках. Укажите один случай, где низкий средний NME не гарантирует хорошего выравнивания: положение глаз может особенно влиять на поворот и масштаб. Не делайте выводов об идентичности людей.

In [ ]:
if RUN_MODEL_TEST:
    picks=[int(np.flatnonzero(POSE_TEST==3)[0]),
           int(np.flatnonzero(POSE_TEST==1)[0]),
           int(np.flatnonzero(GLASSES_TEST==1)[0])]
    pred=TEST_RESULTS[SELECTED_MODEL]["pred"]
    baseline=TEST_RESULTS["mean-shape"]["pred"]
    fig,ax=plt.subplots(3,4,figsize=(15,11))
    for row,i in enumerate(picks):
        im=X_TEST[i];truth=Y_TEST[i]*(IMAGE_SIZE-1)
        estimate=pred[i]*(IMAGE_SIZE-1)
        ax[row,0].imshow(im,cmap="gray",vmin=0,vmax=255)
        ax[row,0].scatter(truth[:,0],truth[:,1],c="lime",s=35,label="GT")
        ax[row,0].scatter(estimate[:,0],estimate[:,1],c="orange",s=35,
                          marker="x",label="модель")
        ax[row,0].set_title(f"Вход: pose={POSE_TEST[i]}, очки={GLASSES_TEST[i]}, NME={errors[i]:.2f}")
        for col,name in enumerate(("mean-shape","coordinate","heatmap"),start=1):
            points=(baseline[i] if name=="mean-shape" else TEST_RESULTS[name]["pred"][i])
            M=weighted_similarity(points*(IMAGE_SIZE-1),TARGET,[1,1,1,0,0])
            aligned=cv2.warpAffine(im,M,(128,128),borderValue=240)
            transformed=transform_points(truth,M)
            ax[row,col].imshow(aligned,cmap="gray",vmin=0,vmax=255)
            ax[row,col].scatter(TARGET[:,0],TARGET[:,1],s=70,facecolors="none",
                                edgecolors="cyan",label="шаблон")
            ax[row,col].scatter(transformed[:,0],transformed[:,1],s=28,c="lime",label="GT после")
            mark=" ★ выбран" if name==SELECTED_MODEL else ""
            ax[row,col].set_title(f"Выравнивание: {name}{mark}")
        for a in ax[row]:a.axis("off")
    ax[0,0].legend(fontsize=7,loc="lower left")
    plt.tight_layout();plt.show()

## Сдача

Выполненный ноутбук с шестью TODO и открытыми проверками; три ручных контроля геометрии; гипотеза и val/test для учебного одного снимка; пилот, полный опыт и абляция аугментации на MTFL; таблица сравнения двух моделей с baseline; выбор по val до официального test; прогноз и фактический test с разбором позы/очков; визуальные примеры выравнивания и вывод в 8–12 предложениях. Укажите источник, seed, устройство, время и версии. Объясните ограничения учебного поднабора, отдельного источника test и неизвестной раздельности личностей. Не добавляйте данные, личные снимки и веса в Git. Критерии — в ASSESSMENT.md.